# Single vs double precision in cuPIQP (float32 vs float64)

CuPIQP supports both **float64** (double, the default) and **float32** (single)
precision. Float32 uses half the memory and is typically faster on GPUs, which makes
it attractive for real-time control or large batches, at the cost of achieving lower precision than float64.

This notebook solves one simple QP in both precisions and shows the trade-off concretely:

- The precision is **fixed when the solver is constructed** (`DenseSolver(dtype=...)`)
  and cannot be changed afterwards.
- The convergence tolerances **default to match the dtype** -- you cannot ask for
  float64-level accuracy from float32 arithmetic.
- Tightening the float32 tolerances does not help: the solver warns, and the residual
  plateaus near the float32 noise floor.

In [1]:
import warnings

import numpy as np
import cupy as cp

from cupiqp import DenseSolver, Settings, Status

## Choosing the precision

Pass `dtype="float32"` or `dtype="float64"` to the solver constructor. The solver's
`Settings` are then initialized with **dtype-appropriate tolerance defaults** (the same
ones you get from `Settings.for_dtype(dtype)`). float32 defaults are looser by several
orders of magnitude, because single-precision arithmetic simply has fewer significant
digits (about 7, vs about 16 for double).

In [2]:
for dt in ("float64", "float32"):
    settings = Settings.for_dtype(dt)
    print(f"{dt}: eps_abs={settings.eps_abs:.0e}  eps_rel={settings.eps_rel:.0e}  "
          f"duality_gap_abs={settings.eps_duality_gap_abs:.0e}")

float64: eps_abs=1e-08  eps_rel=1e-09  duality_gap_abs=1e-08
float32: eps_abs=1e-04  eps_rel=1e-04  duality_gap_abs=1e-04


## A simple QP

A small box-constrained QP

$$
\min_{x}\ \tfrac{1}{2}\, x^\top P x + c^\top x
\quad\text{s.t.}\quad -1 \le x \le 1,
$$

with `P` symmetric positive definite. We build the data once in float64; each solver
casts it internally to its own dtype, so the same arrays feed both runs.

In [3]:
rng = np.random.default_rng(0)
n = 100
M = rng.standard_normal((n, n))

P = cp.asarray(M.T @ M + np.eye(n))      # symmetric positive definite
c = cp.asarray(rng.standard_normal(n))
x_l = cp.full(n, -1.0)
x_u = cp.full(n, 1.0)

In [4]:
def summarize(tag, result, x_ref=None):
    """Print status, iteration count, and final residuals for one solve."""
    info = result.info
    line = (f"{tag:>22}: status={info.status[0].name:<22} "
            f"iters={int(info.iter[0]):>3}  "
            f"primal_res={float(info.primal_res[0]):.2e}  "
            f"dual_res={float(info.dual_res[0]):.2e}  "
            f"gap={float(info.duality_gap[0]):.2e}")
    if x_ref is not None:
        err = float(cp.linalg.norm(result.x[0].astype(cp.float64) - x_ref))
        line += f"  ||x - x_ref||={err:.2e}"
    print(line)

## Solve in float64 (the high-precision reference)

Double precision converges to very small residuals -- a duality gap around `1e-9` and
primal/dual residuals near machine-level. We keep its solution `x_ref` as the reference
to measure the float32 solution against.

In [5]:
solver64 = DenseSolver(dtype="float64")
solver64.setup(P=P, c=c, x_l=x_l, x_u=x_u)
solver64.solve()

x_ref = solver64.result.x[0].copy()      # (n,) float64 reference solution
summarize("float64", solver64.result)

               float64: status=CUPIQP_SOLVED          iters=  7  primal_res=4.20e-16  dual_res=2.61e-13  gap=1.03e-10


## Solve in float32

Single precision converges in a few iterations to its (looser) default tolerances. The
residuals settle several orders of magnitude higher than float64, and the solution
differs from the float64 reference by roughly `1e-4`. **That gap is the accuracy ceiling
of float32** -- not a bug, just the limit of ~7 significant digits.

In [6]:
solver32 = DenseSolver(dtype="float32")
solver32.setup(P=P, c=c, x_l=x_l, x_u=x_u)
solver32.solve()

summarize("float64 (reference)", solver64.result)
summarize("float32", solver32.result, x_ref=x_ref)

   float64 (reference): status=CUPIQP_SOLVED          iters=  7  primal_res=4.20e-16  dual_res=2.61e-13  gap=1.03e-10
               float32: status=CUPIQP_SOLVED          iters=  4  primal_res=1.94e-07  dual_res=7.96e-06  gap=9.16e-05  ||x - x_ref||=1.22e-05


## float32 cannot be pushed to float64 precision

It is tempting to just tighten the float32 tolerances to `1e-8` and ask for more
accuracy. cuPIQP warns when you set a tolerance below the float32 recommended floor,
because single precision cannot resolve residuals that small. The solve then runs out of
iterations while the residual **plateaus near the float32 noise floor** -- it never
reaches the requested tolerance.

In [7]:
solver32_tight = DenseSolver(dtype="float32")

# Asking float32 for float64-level accuracy. This triggers a warning.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    solver32_tight.settings.eps_abs = 1e-8
    solver32_tight.settings.eps_rel = 1e-8
for w in caught:
    print("WARNING:", w.message)

solver32_tight.setup(P=P, c=c, x_l=x_l, x_u=x_u)
solver32_tight.solve()
print()
summarize("float32 (eps=1e-8)", solver32_tight.result, x_ref=x_ref)


    float32 (eps=1e-8): status=CUPIQP_MAX_ITER_REACHED iters=249  primal_res=2.05e-07  dual_res=7.96e-06  gap=6.68e-06  ||x - x_ref||=6.25e-06


The status comes back as `CUPIQP_MAX_ITER_REACHED`: float32 burned through every
iteration without hitting `1e-8`, and the residual stalled near `1e-7`. Tightening the
tolerance did not buy float64 accuracy -- it only cost iterations.

## Solve time: float32 vs float64

The reason to accept float32's lower accuracy is **speed**: single precision moves half
the bytes and runs faster GPU arithmetic. To time the per-solve GPU cost fairly we:

- use a larger, **batched** problem so the GPU (not Python overhead) is the bottleneck,
- **warm up** first -- the very first solve compiles the Warp kernels and captures the
  CUDA graph, which we do not want in the timing,
- **synchronize** the device around a loop of repeated solves and report the mean.

float32 is faster for two distinct reasons: each iteration is cheaper (fewer bytes,
faster math), **and** at its looser tolerances it converges in fewer iterations. The
per-solve time mixes both effects, so we also report the **per-iteration** time, which
isolates the cost of the arithmetic itself -- the fairer apples-to-apples comparison.

Absolute numbers depend on your GPU; the **ratios** are the takeaway.

In [8]:
import time


def make_batch(n, B, seed=0):
    rng = np.random.default_rng(seed)
    M = rng.standard_normal((B, n, n))
    P = cp.asarray(np.einsum("bij,bik->bjk", M, M) + np.eye(n)[None])  # SPD per batch
    c = cp.asarray(rng.standard_normal((B, n)))
    x_l = cp.full((B, n), -1.0)
    x_u = cp.full((B, n), 1.0)
    return P, c, x_l, x_u


def benchmark(dtype, P, c, x_l, x_u, warmup=3, reps=20):
    """Mean GPU time per solve (ms) and iteration count, after warmup, with sync."""
    solver = DenseSolver(dtype=dtype)
    solver.setup(P=P, c=c, x_l=x_l, x_u=x_u)
    for _ in range(warmup):                 # compile kernels + capture CUDA graph
        solver.solve()
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    for _ in range(reps):
        solver.solve()
    cp.cuda.Device().synchronize()
    ms = (time.perf_counter() - start) / reps * 1e3
    return ms, int(solver.result.info.iter[0])


n_bench, B_bench = 128, 64
Pb, cb, xlb, xub = make_batch(n_bench, B_bench)
print(f"batched dense QP: B={B_bench}, n={n_bench}\n")

ms64, it64 = benchmark("float64", Pb, cb, xlb, xub)
ms32, it32 = benchmark("float32", Pb, cb, xlb, xub)

# float32 also converges in fewer iterations, so divide out the iteration count to
# compare the cost of one iteration -- this isolates the arithmetic/bandwidth gain.
ms_per_iter64 = ms64 / it64
ms_per_iter32 = ms32 / it32

print(f"float64: {ms64:6.2f} ms/solve  /  {it64} iters  =  {ms_per_iter64:.3f} ms/iter")
print(f"float32: {ms32:6.2f} ms/solve  /  {it32} iters  =  {ms_per_iter32:.3f} ms/iter")
print()
print(f"per-solve speedup:     {ms64 / ms32:.2f}x")
print(f"per-iteration speedup: {ms_per_iter64 / ms_per_iter32:.2f}x")

batched dense QP: B=64, n=128

float64:   8.42 ms/solve  /  7 iters  =  1.203 ms/iter
float32:   4.43 ms/solve  /  5 iters  =  0.887 ms/iter

per-solve speedup:     1.90x
per-iteration speedup: 1.36x


## Summary -- when to use which

- **float64** (default) -- use it when you need accurate solutions: tight residuals,
  reliable convergence, sensitivity/gradient computations.
- **float32** -- use it when a lower-precision answer (about `1e-3` to `1e-4`) is good
  enough and you want the speed (often ~1.5x-2.5x faster for medium-to-large problems)
  and the halved memory: real-time MPC, RL rollouts, or large batches where throughput
  matters more than the last few digits.
- The dtype is fixed at construction and its tolerances default to match; do not try to
  force float32 below its precision floor.